# Initialization

In [0]:
import pyspark.sql.functions as F

In [0]:
ERP_LOCATION_RENAME_MAP = {
    "cid": "customer_id",
    "cntry": "country"
}

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

# Transformations

## Renaming the columns

In [0]:
for old_name, new_name in ERP_LOCATION_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Removing '-' symbol

In [0]:
df = df.withColumn(
    "customer_id",
    F.regexp_replace(
        F.col("customer_id"),
        "-",
        ""
    )
)

## Country Standardization

In [0]:
df = df.withColumn(
    "country",
    F.when(
        F.upper(F.trim(F.col("country"))).isin("US", "USA", "UNITED STATES"),
        "United States"
    )
    .when(
        F.upper(F.trim(F.col("country"))).isin("DE", "GERMANY"),
        "Germany"
    )
    .when(
        (F.trim(F.col("country")) == "") |
        (F.col("country").isNull()),
        "Unknown"
    )
    .otherwise(
        F.trim(F.col("country"))
    )
)

## Selecting the columns

In [0]:
df = df.select(
    "customer_id",
    "country"
)

# Writing into Silver table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("Silver.erp_locations")
)